# What this file does
- Add Company nodes
- Apply Louvain to Company nodes and write the community id back to Company nodes
- Add corp_community_id property to Complaint nodes based on Companies' community ID

# Dependencies
#### Run the following file(s) before running this code.
- 03_baseline_similarity_graph.ipynb (or 03b or 03c)
- 07_gds_Louvain_Summary.ipynb
- 21_gds_Centrality_on-graph.ipynb

##### To add some nodes, this code is updating the data in Mongo, and then updating neo4j accordingly.

##### Note:
URL of the Neo4j browser:
- https://[IP address]:7473/browser/

ID & Pass: 
- Use the one in .env


In [1]:
# Config
DB_NAME:str         = "project03"
COLLECTION_NAME:str = "complaints"

In [2]:
import time
from datetime import datetime, timedelta

In [3]:
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
import tiktoken
import textwrap
import logging

# pretty print some of the Neo4j outputs
logger = logging.getLogger("neo4j")
logger.setLevel(logging.CRITICAL)

In [4]:
from dotenv import load_dotenv  
load_dotenv()

True

In [5]:
# from openai import OpenAI
import neo4j
from pymongo import MongoClient

In [6]:
# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

In [7]:
warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [8]:
# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows
pd.set_option('display.max_rows', None)

In [9]:
# Timestamp (Start)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

start_time = time.time()

2026-01-08_22:23:33


### Mongo

In [10]:
# Connect to the mongo database
# mongodb is the protocol; mongo is the hostname, which for us is the container name; 27017 is the TCP port number
mongo = MongoClient("mongodb://mongo:27017/")

In [11]:
# # Drop the database
# mongo.drop_database(DB_NAME)

In [12]:
# Access to the existing mongo database (this seems to be the same command with creating a new database)
db = mongo[DB_NAME]

In [13]:
# Connect to the existing collection (this seems to be the same command with creating a new collection)
collection = db[COLLECTION_NAME]

In [14]:
# An example of the data in mongo
collection.find_one()

{'_id': ObjectId('69609e65d00a36fd6bcd31e2'),
 'date_received': '06/05/25',
 'product': 'Credit reporting or other personal consumer reports',
 'sub_product': 'Credit reporting',
 'issue': 'Incorrect information on your report',
 'sub_issue': 'Information belongs to someone else',
 'consumer_complaint_narrative': 'I hope this complaint finds you well. I would like to follow up on the complaints I sent to your attention regarding certain items on my credit report. As of today, I have not received any communication or updates regarding the resolution of the disputes I sent to the bureaus. I understand that the investigation process takes time, but I would like to inquire about the current status and any progress made in resolving the issues outlined in my previous communication. The following are the items that I would like to be removed from my credit file. //',
 'company_public_response': 'Company has responded to the consumer and the CFPB and chooses not to provide a public response',

In [15]:
# Obtain all data in DataFrame
cursor = collection.find({}, {"_id": 0})
df = pd.DataFrame(cursor)

### Neo4j

In [16]:
driver = neo4j.GraphDatabase.driver(
    uri=os.environ.get("NEO4J_URI"), 
    auth=(os.environ.get("NEO4J_USERNAME"), 
          os.environ.get("NEO4J_PASSWORD"))
)

In [17]:
session = driver.session(database="neo4j")

In [18]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [19]:
def my_neo4j_create_node(node_label, node_name):
    "create a node with label node_type"

    # Note1: Cannot use parameter in labels (so, use f-string, instead).
    #        When using f-string, need to double {} when using it as a part of Cypher.
    # Note2: Merge: Create if not exists, otherwise reuse
    query = f"""
    
    MERGE (:{node_label} {{name: $node_name}})
    
    """
    
    session.run(query, node_name=node_name)
    

In [20]:
# Create constraints
query = """
CREATE CONSTRAINT company_name_unique IF NOT EXISTS
FOR (c:Company)
REQUIRE c.name IS UNIQUE;
"""

session.run(query)

# Add Company nodes and relationship

In [21]:
# Create nodes
for document in collection.find():
    company = document.get("company")
    my_neo4j_create_node("Company", company)

In [22]:
# Create relationships
for document in collection.find():

    from_node = document.get("complaint_id")   # This is a Complaint node
    to_node = document.get("company")          # This is a Company node
    weight = 1
    
    query = """
    
    MATCH (from: Complaint), 
          (to: Company)
    WHERE from.complaint_id = $from_node and to.name = $to_node
    MERGE (from)-[:COMPLAIN_TO {weight: $weight}]->(to)
    
    """

    session.run(query, {"from_node":from_node, "to_node":to_node, "weight":weight})

# Add Relationship between Company and Categories

In [23]:
query = """
MATCH (corp:Company)<-[:COMPLAIN_TO]-(comp:Complaint)
    -[:IN_CATEGORY]->(cat:Category)

// Note:
// - pairCount: Number of complaints
WITH corp, cat,
     count(comp) AS pairCount

MERGE (cat)-[r:COMPLAINS_TO]->(corp)
SET r.num = pairCount;

"""

session.run(query)

In [24]:
# Drop the in-memory graph
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

# Create an in-memory graph named 'ds_graph'
# Nodes: Category, Company
# Relationship: COMPLAINS_TO, weighted by r.num
query = """
CALL gds.graph.project(
    'ds_graph', 
    ['Category', 'Company'], 
    {
        COMPLAINS_TO: {
            type: 'COMPLAINS_TO', 
            orientation: 'NATURAL',
            properties: 'num'
        }
    }
)
"""

session.run(query)

### Company Counts

In [25]:
Company_counts = df['company'].value_counts()
print(Company_counts)

company
TRANSUNION INTERMEDIATE HOLDINGS, INC.                                           69
EQUIFAX, INC.                                                                    65
Experian Information Solutions Inc.                                              61
CITIBANK, N.A.                                                                    3
CAPITAL ONE FINANCIAL CORPORATION                                                 3
Resurgent Capital Services L.P.                                                   3
Chime Financial Inc                                                               3
CL Holdings LLC                                                                   2
MOHELA                                                                            2
BANK OF AMERICA, NATIONAL ASSOCIATION                                             2
I.C. System, Inc.                                                                 2
WELLS FARGO & COMPANY                                               

### Degree Centrality

In [26]:
# Note: 'UNDIRECTED': Count both in and out going arrows
query = """
CALL gds.degree.stream(
    'ds_graph',
    {orientation: 'UNDIRECTED'}
)
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
WHERE n:Company
RETURN n.name AS Company, score AS Degree
ORDER BY Degree DESC, Company;
"""

my_neo4j_run_query_pandas(query).head(10)
# session.run(query)

,Company,Degree
0,"EQUIFAX, INC.",10.0
1,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",9.0
2,Experian Information Solutions Inc.,8.0
3,CAPITAL ONE FINANCIAL CORPORATION,3.0
4,"CITIBANK, N.A.",2.0
5,CL Holdings LLC,2.0
6,"I.C. System, Inc.",2.0
7,Resurgent Capital Services L.P.,2.0
8,WELLS FARGO & COMPANY,2.0
9,AMERICAN EXPRESS COMPANY,1.0


# Louvain

In [27]:
# Get the results from the Louvain community detection algorithm stored in DataFrame
query = """

CALL gds.louvain.stream(
    'ds_graph',
    {
        includeIntermediateCommunities: true,
        relationshipWeightProperty: 'num'
    }
)
YIELD nodeId, communityId, intermediateCommunityIds
WITH gds.util.asNode(nodeId) AS n, communityId, intermediateCommunityIds
WHERE n:Company
RETURN 
    n.name AS company,
    communityId AS community,
    intermediateCommunityIds AS intermediate_community
ORDER BY community, company

"""

community_df = my_neo4j_run_query_pandas(query)

In [28]:
community_df.head(20)

,company,community,intermediate_community
0,SinglePoint GI,15,"[15, 15, 15, 15]"
1,"Community Loan Servicing, LLC (formerly known ...",16,"[16, 16, 16, 16]"
2,"CITIBANK, N.A.",17,"[13, 13, 13, 17]"
3,Resurgent Capital Services L.P.,17,"[24, 24, 13, 17]"
4,SYNCHRONY FINANCIAL,17,"[17, 17, 17, 17]"
5,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",17,"[10, 24, 13, 17]"
6,Solar Mosaic LLC,18,"[18, 18, 18, 18]"
7,Fig Tech Inc.,19,"[19, 19, 19, 19]"
8,WELLS FARGO & COMPANY,22,"[22, 22, 22, 22]"
9,"Portfolio Recovery Associates, LLC",23,"[23, 23, 23, 23]"


In [29]:
# Write back the results of the Louvain to Knowledge Graph
# (Community ID)

query = """

CALL gds.louvain.write(
    'ds_graph',
    {
        includeIntermediateCommunities: false,
        relationshipWeightProperty: 'num',
        writeProperty: 'community_id'
    }
)
YIELD communityCount, nodePropertiesWritten

"""

session.run(query)

In [30]:
# Write back the results of the Louvain to Knowledge Graph
# (Intermediate Community ID)

query = """

CALL gds.louvain.write(
    'ds_graph',
    {
        includeIntermediateCommunities: true,
        relationshipWeightProperty: 'num',
        writeProperty: 'intermediate_community'
    }
)
YIELD communityCount, nodePropertiesWritten

"""

session.run(query)

In [31]:
# Add properties to Complaint nodes based on Companies' community ID

query = """
MATCH (comp:Complaint)-[:COMPLAIN_TO]->(corp:Company)

SET
    comp.corp_community_id = corp.community_id,
    comp.corp_intermediate_community = corp.intermediate_community

RETURN count(comp) AS complaintsUpdated;

"""

session.run(query)

In [32]:
# Timestamp (End)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

end_time = time.time()
elapsed_seconds = end_time - start_time

# Convert elapsed seconds to minutes and seconds
minutes = int(elapsed_seconds // 60)
seconds = elapsed_seconds % 60

print(f"Program elapsed time: {minutes} minutes and {seconds:.2f} seconds")

2026-01-08_22:23:37
Program elapsed time: 0 minutes and 3.43 seconds
